# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [4]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ["stores", "sales", "vendor"], # ENTER A TABLE COLUMNS HERE,
        'definition': ["stores information, name, location...", "sales details per month","company vendors details ..."]} # ENTER A TABLE DEFINITATIONS HERE
df = pd.DataFrame(data)
print(df)


    table                             definition
0  stores  stores information, name, location...
1   sales                sales details per month
2  vendor            company vendors details ...


In [5]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [6]:
print(text_tables)

stores: stores information, name, location...
sales: sales details per month
vendor: company vendors details ...


In [17]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Question:
{question}
"""


In [18]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Give me the vendors information and sales") #ENTER YOUR QUERY HERE

In [19]:
print(return_OAI(pqt1))

```json
{
    "tables": ["sales", "vendor"]
}
```


In [20]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question= "Show me the biggest store")#ENTER YOUR QUERY HERE)

In [21]:
print(return_OAI(pqt3))

```json
{
    "tables": ["stores"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [24]:
# E-commerce tables
ecommerce_data = {
    'table': ["customers", "products", "orders", "reviews", "categories"],
    'definition': [
        "customer information including ID, name, email, registration date, shipping address",
        "product catalog with ID, name, price, description, stock quantity, category ID",
        "order records with order ID, customer ID, product ID, quantity, order date, status",
        "product reviews with review ID, product ID, customer ID, rating, comment, review date",
        "product categories with category ID, category name, parent category, description"
    ]
}
ecommerce_df  = pd.DataFrame(ecommerce_data)
print(df)

    table                             definition
0  stores  stores information, name, location...
1   sales                sales details per month
2  vendor            company vendors details ...


In [25]:
questions_ecommerce = [
    "What are the top 5 best-selling products this month?",
    "Which customers have placed orders worth more than $500?",
    "What's the average rating for products in the 'Electronics' category?",
    "How many orders are still pending delivery?",
    "Which products are running low on stock (less than 10 units)?"
]

In [26]:
for i in enumerate(questions_ecommerce):
    print(return_OAI(prompt_question_tables.format(tables=ecommerce_data, question=i)))

```json
["products", "orders"]
```
```json
["customers", "orders"]
```
```json
["products", "reviews", "categories"]
```
```json
["orders"]
```
```json
["products"]
```


In [28]:
# Example 2: Hospital Management System
hospital_data = {
    'table': ["patients", "doctors", "appointments", "departments", "treatments"],
    'definition': [
        "patient records with patient ID, name, age, gender, contact info, medical history",
        "doctor information including doctor ID, name, specialization, department, contact details",
        "appointment scheduling with appointment ID, patient ID, doctor ID, date, time, status",
        "hospital departments with department ID, name, head doctor, location, capacity",
        "treatment records with treatment ID, patient ID, doctor ID, diagnosis, medication, cost"
    ]
}
hospital_df = pd.DataFrame(hospital_data)
print(hospital_df)

          table                                         definition
0      patients  patient records with patient ID, name, age, ge...
1       doctors  doctor information including doctor ID, name, ...
2  appointments  appointment scheduling with appointment ID, pa...
3   departments  hospital departments with department ID, name,...
4    treatments  treatment records with treatment ID, patient I...


In [29]:
questions_hospital = [
    "How many appointments does Dr. Smith have scheduled for next week?",
    "What's the average age of patients in the Cardiology department?",
    "Which treatments cost more than $1000?",
    "How many patients visited the Emergency department last month?",
    "What are the most common diagnoses recorded this year?"
]

In [30]:
for i in enumerate(questions_hospital):
    print(return_OAI(prompt_question_tables.format(tables=hospital_data, question=i)))

```json
["doctors", "appointments"]
```
```json
["patients", "departments"]
```
```json
["treatments"]
```
```json
["patients", "appointments", "departments"]
```
```json
["appointments", "treatments"]
```


In [31]:
# Example 3: Educational Institution Database
education_data = {
    'table': ["students", "courses", "instructors", "enrollments", "grades"],
    'definition': [
        "student information with student ID, name, major, year, GPA, contact details",
        "course catalog with course ID, course name, credits, prerequisites, semester offered",
        "instructor details including instructor ID, name, department, office location, email",
        "enrollment records with enrollment ID, student ID, course ID, semester, enrollment date",
        "grade records with grade ID, student ID, course ID, assignment type, score, grade letter"
    ]
}

In [32]:
questions_education = [
    "What's the average GPA of Computer Science majors?",
    "Which courses have the highest enrollment this semester?",
    "How many students are failing Math 101?",
    "Which instructor teaches the most courses?",
    "What percentage of students completed their prerequisites for Advanced Physics?"
]

In [33]:
for i in enumerate(questions_education):
    print(return_OAI(prompt_question_tables.format(tables=education_data, question=i)))

```json
["students"]
```
```json
["courses", "enrollments"]
```
{
    "tables": ["students", "courses", "enrollments", "grades"]
}
{
    "tables": ["instructors", "courses"]
}
{
    "tables": ["students", "courses", "enrollments"]
}


# Feedback

This lab was pretty straightforward, but it was a good exercise to see how well GPT-3.5 can understand and generate SQL queries based on table names and descriptions.